# 03. 전처리와 분석 마트

SQL 기준은 [`sql/mart_overview.sql`](../sql/mart_overview.sql)이다. 마트 생성문은 구조를 설명하기 위해 제시한다. 이 노트북은 **읽기 전용 검산 캐시**만 사용하며 마트를 다시 만들지 않는다.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'cache_context.json').is_file() and (p / 'sql').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from query_cache import QueryCache

load_dotenv(PROJECT_ROOT / '.env')
required_env = ('DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_PORT', 'DB_NAME')
if any(not os.getenv(k) for k in required_env):
    raise RuntimeError('캐시 출처 확인용 .env 항목이 부족하다.')
# Engine은 DB 출처 식별에만 사용한다. 캐시 조회는 DB에 연결하지 않는다.
engine = create_engine(URL.create(
    'mysql+pymysql', username=os.environ['DB_USER'], password=os.environ['DB_PASSWORD'],
    host=os.environ['DB_HOST'], port=int(os.environ['DB_PORT']), database=os.environ['DB_NAME'],
))
COMPATIBILITY_FILE = PROJECT_ROOT / 'sql' / 'cache_compatibility.json'

FINAL_CACHE = QueryCache(
    engine=engine, sql_file=PROJECT_ROOT / 'sql' / 'mart_overview.sql',
    upstream_sql_files=(), cache_dir=PROJECT_ROOT / 'cache',
    context_file=PROJECT_ROOT / 'cache_context.json',
)
def read_mart(name, **kwargs):
    return FINAL_CACHE.read_compatible_cached(
        name, compatibility_file=COMPATIBILITY_FILE, **kwargs
    )


## 1. 전처리 핵심

- 유효 세션은 `user_session`이 있고, 세션 안의 `user_id`가 하나이며, 시작부터 종료까지 1일 이하인 방문이다.
- 이벤트 카운트에서 `price < 0`을 제외하고 0원은 포함한다. 반복 이벤트는 임의로 삭제하지 않는다.
- 순차 퍼널은 동일 세션 안에서 `view < cart < purchase`의 엄격한 시각 순서를 요구한다. 사용자×상품 구매 여정은 이후 여러 세션을 연결한다.

세션의 전체 시작·종료 시각으로 유효성을 판정하고, 카운트에는 0원 이벤트를 포함한다. 이 규칙을 적용한 마트의 구조와 행 수를 아래에서 확인한다.

## 2. 파이프라인

`월별 CSV → events → 품질 점검·유효 세션 → mart_session → mart_session_product → mart_user_product_session → 사용자×상품·30일 대표 첫 구매 → 최초 cart 시간·실험 적격군`

세 마트는 서로 다른 관찰 단위를 제공한다. 30일 구매 여정과 최초 cart 시간은 영구 마트가 아니라 EDA 단계에서 **사용자×상품 단위로 계산하는 분석 집계**다.

## 3. 마트 구조

| 마트 | 1행의 의미 (grain) | 목적 |
|---|---|---|
| `mart_session` | 유효 `user_session` 1개 | 방문 안의 순차 행동과 세션 퍼널 |
| `mart_session_product` | 유효 `user_session × product_id` 1개; 조회·담기·구매 중 하나 포함 | 같은 상품의 방문 내 행동 |
| `mart_user_product_session` | 유효 `user_session × product_id` 1개; `user_id`와 행동 시각 보존, remove-only 포함 | 첫 구매 전 행동과 이전 세션 연결 |

`mart_user_product_session`의 이름은 사용자×상품 집계처럼 보이지만 실제 PK는 **세션×상품**이다. 마트 생성 정의는 [`sql/mart_overview.sql`](../sql/mart_overview.sql)에 있다.

## 4. 마트 기본 검산

검증된 캐시에서 저장 행 수와 기본키 중복 여부를 확인한다. 전체 원본과 대조하는 고비용 검산은 반복하지 않는다.

In [2]:
inventory = read_mart('mart_inventory')
ms = read_mart('mart_session_reconciliation').iloc[0]
msp = read_mart('mart_session_product_reconciliation').iloc[0]
mups = read_mart('mart_user_product_session_reconciliation').iloc[0]
expected = {'mart_session':4_499_479,'mart_session_product':12_102_048,'mart_user_product_session':13_385_787}
rows = inventory.set_index('마트')['실제_행수'].astype('int64').to_dict()
assert rows == expected
assert int(ms['기본키_중복']) == int(msp['복합키_중복']) == int(mups['복합키_중복']) == 0
assert rows['mart_session'] == int(ms['마트_행수'])
assert rows['mart_session_product'] == int(msp['마트_행수'])
assert rows['mart_user_product_session'] == int(mups['마트_행수'])
display(pd.DataFrame({
    '마트':list(expected), '저장 행 수':[rows[k] for k in expected],
    '키 중복':[int(ms['기본키_중복']),int(msp['복합키_중복']),int(mups['복합키_중복'])],
    '검산한 키':['user_session','user_session × product_id','user_session × product_id'],
}))


[mart_inventory] cache HIT 5e70190b19e2 · 3행
[mart_session_reconciliation] cache HIT 7fc731e0d5b3 · 1행
[mart_session_product_reconciliation] cache HIT 33910b1d924e · 1행
[mart_user_product_session_reconciliation] cache HIT 33e6978d30c4 · 1행


,마트,저장 행 수,키 중복,검산한 키
0,mart_session,4499479,0,user_session
1,mart_session_product,12102048,0,user_session × product_id
2,mart_user_product_session,13385787,0,user_session × product_id


## 5. 요약

세션 퍼널의 분모는 `mart_session`의 유효 방문이다. 같은 상품의 구매 세션 경로는 `mart_user_product_session`에서 세션×상품을 세고, 여러 방문의 대표 첫 구매와 최초 cart 분석은 별도 사용자×상품 집계를 사용한다. 마트 행 수끼리 나눈 값은 전환율이 아니다. [04 EDA](04_eda.ipynb)에서 질문에 맞춰 단위를 바꾼다.